# Fase II — Modelado Cinemático del Robot Icebot

**Objetivo:** Establecer los sistemas de coordenadas, obtener la tabla de parámetros
Denavit-Hartenberg (DH), y desarrollar las matrices de transformación homogénea
para el robot Icebot a partir de su modelo URDF.

**Estructura del robot (del URDF):**

| # | Joint | Tipo | Padre → Hijo | Eje |
|---|-------|------|-------------|-----|
| 1 | axis_0 | Revoluta | base_link → link_1 | (0,0,1) |
| 2 | axis_1 | Revoluta | link_1 → link_2 | (0,0,1) |
| 3 | axis_2 | Revoluta | link_2 → link_z_rot | (0,0,1) |
| 4 | axis_3 | Prismática | link_z_rot → end_effector | (0,0,-1) |
| 5 | tcp_joint | Fija | end_effector → tool0 | — |

Todas las juntas tienen ejes paralelos (eje Z), lo que hace del Icebot un robot
tipo SCARA de 3 GDL rotacionales + 1 prismático vertical.

---
## 1. Asignación de Sistemas de Coordenadas

Siguiendo la convención **Denavit-Hartenberg (DH) estándar**:

- $z_{i-1}$: eje del joint $i$
- $x_i$: perpendicular común a $z_{i-1}$ y $z_i$ (dirección del eslabón)
- $y_i$: completa el sistema dextrógiro

### Asignación

| Frame | Descripción |
|-------|-------------|
| {0} | **Base**: origen en la base del robot. $z_0$ vertical (eje de axis_0). $x_0$ hacia adelante. |
| {1} | **Link 1**: $z_1 \parallel z_0$ (eje de axis_1). $x_1$ apunta hacia link_1. Origen a altura $H_1$ desde la base. |
| {2} | **Link 2**: $z_2 \parallel z_1$ (eje de axis_2). $x_2$ a lo largo de link_1. Origen a distancia $L_1$ de {1}. |
| {3} | **Link Z Rot**: $z_3 \parallel z_2$ (eje de axis_3, prismática). $x_3$ a lo largo de link_2. Origen a distancia $L_2$ de {2}. |
| {4} | **End Effector**: $z_4 \parallel z_3$. $x_4 \parallel x_3$. Origen a distancia $d_4$ de {3}, donde $d_4 = H_2 - q_4$. |
| {T} | **Tool tip / tool0**: punta del efector final, a 0.120 m del origen de {4} en dirección $-z$. Corresponde al frame `tool0` en el URDF, conectado mediante un joint fijo `tcp_joint`. |

**Constantes geométricas del URDF:**

- $H_1 = 0.100\,\text{m}$ — altura del primer joint rotacional
- $L_1 = 0.125\,\text{m}$ — longitud del eslabón 1
- $L_2 = 0.100\,\text{m}$ — longitud del eslabón 2
- $H_2 = 0.060\,\text{m}$ — altura base del joint prismático
- $\text{Tool} = 0.120\,\text{m}$ — longitud del efector final

**Variables articulares:**

- $\theta_1, \theta_2, \theta_3$: variables de las juntas revolutas (rad)
- $q_4$: variable de la junta prismática (m), $0 \leq q_4 \leq 0.060$, dirección positiva = hacia abajo (negativa en Z según URDF)

---
## 2. Tabla de Parámetros Denavit-Hartenberg

Convención DH estándar:
- $a_i$: distancia de $z_{i-1}$ a $z_i$ medida sobre $x_i$
- $\alpha_i$: ángulo de $z_{i-1}$ a $z_i$ medido sobre $x_i$
- $d_i$: distancia de $x_{i-1}$ a $x_i$ medida sobre $z_{i-1}$
- $\theta_i$: ángulo de $x_{i-1}$ a $x_i$ medido sobre $z_{i-1}$

| $i$ | $a_i$ (m) | $\alpha_i$ (rad) | $d_i$ (m) | $\theta_i$ (rad) |
|-----|-----------|------------------|-----------|-------------------|
| 1 | 0 | 0 | $H_1 = 0.100$ | $\theta_1$ (var) |
| 2 | $L_1 = 0.125$ | 0 | 0 | $\theta_2$ (var) |
| 3 | $L_2 = 0.100$ | 0 | 0 | $\theta_3$ (var) |
| 4 | 0 | 0 | $H_2 - q_4$ | 0 |

Nota: El joint prismático (axis_3) tiene eje $(0,0,-1)$ en URDF, por lo que su
desplazamiento en $z$ es $H_2 - q_4$ (la dirección positiva de $q_4$ extiende el
efector hacia abajo, restando altura).

In [1]:
from sympy import symbols, Matrix, cos, sin, simplify, pi, latex

# Variables articulares
theta1, theta2, theta3 = symbols("theta1 theta2 theta3", real=True)
q4 = symbols("q4", real=True, nonnegative=True)  # prismática: 0 ≤ q4 ≤ 0.060

# Constantes geométricas del URDF
H1 = 0.100    # altura del joint 1 (axis_0)
L1 = 0.125    # longitud del eslabón 1
L2 = 0.100    # longitud del eslabón 2
H2 = 0.060    # altura base del joint prismático (axis_3)
Tool = 0.120  # longitud del efector final

In [2]:
# ---------------------------------------------------------------------------
# Matriz de transformación homogénea DH estándar
#   T = Rot_z(theta) * Trans_z(d) * Trans_x(a) * Rot_x(alpha)
# ---------------------------------------------------------------------------
def dh(a, alpha, d, theta):
    """Matriz DH estándar 4x4."""
    ct = cos(theta)
    st = sin(theta)
    ca = cos(alpha)
    sa = sin(alpha)
    return Matrix([
        [ct, -st*ca,  st*sa,  a*ct],
        [st,  ct*ca, -ct*sa,  a*st],
        [0,   sa,     ca,     d],
        [0,   0,      0,      1]
    ])

---
## 3. Matrices de Transformación Individuales

Aplicando la tabla de parámetros DH a la función $\text{dh}(a_i, \alpha_i, d_i, \theta_i)$:

In [3]:
# Matriz A1: base_link → link_1 (joint 1, revolute, eje Z)
A1 = dh(a=0,     alpha=0, d=H1,      theta=theta1)

# Matriz A2: link_1 → link_2 (joint 2, revolute, eje Z)
A2 = dh(a=L1,    alpha=0, d=0,       theta=theta2)

# Matriz A3: link_2 → link_z_rot (joint 3, revolute, eje Z)
A3 = dh(a=L2,    alpha=0, d=0,       theta=theta3)

# Matriz A4: link_z_rot → end_effector (joint 4, prismática, eje Z)
#   URDF axis=(0,0,-1) → d4 = H2 - q4
A4 = dh(a=0,     alpha=0, d=H2 - q4, theta=0)

# --- Mostramos cada matriz ---
print("A1 — Transformación base → link_1:")
display(A1)

print("A2 — Transformación link_1 → link_2:")
display(A2)

print("A3 — Transformación link_2 → link_z_rot:")
display(A3)

print("A4 — Transformación link_z_rot → end_effector:")
display(A4)

A1 — Transformación base → link_1:


Matrix([
[cos(theta1), -sin(theta1), 0,   0],
[sin(theta1),  cos(theta1), 0,   0],
[          0,            0, 1, 0.1],
[          0,            0, 0,   1]])

A2 — Transformación link_1 → link_2:


Matrix([
[cos(theta2), -sin(theta2), 0, 0.125*cos(theta2)],
[sin(theta2),  cos(theta2), 0, 0.125*sin(theta2)],
[          0,            0, 1,                 0],
[          0,            0, 0,                 1]])

A3 — Transformación link_2 → link_z_rot:


Matrix([
[cos(theta3), -sin(theta3), 0, 0.1*cos(theta3)],
[sin(theta3),  cos(theta3), 0, 0.1*sin(theta3)],
[          0,            0, 1,               0],
[          0,            0, 0,               1]])

A4 — Transformación link_z_rot → end_effector:


Matrix([
[1, 0, 0,         0],
[0, 1, 0,         0],
[0, 0, 1, 0.06 - q4],
[0, 0, 0,         1]])

### Matriz de la herramienta (Tool)

El efector final es un cilindro de 0.120 m de largo, con origen visual en
$(0,0,-0.06)$. La punta está a 0.120 m del origen del frame {4} en dirección $-z$:

In [4]:
Ttool = Matrix([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, -Tool],
    [0, 0, 0, 1]
])

print("Ttool — Transformación end_effector → tool tip:")
display(Ttool)

Ttool — Transformación end_effector → tool tip:


Matrix([
[1, 0, 0,     0],
[0, 1, 0,     0],
[0, 0, 1, -0.12],
[0, 0, 0,     1]])

---
## 4. Matriz de Transformación Homogénea Total

La cinemática directa se obtiene multiplicando en orden las transformaciones:

$$T_{\text{base}}^{\text{tool}} = A_1 \cdot A_2 \cdot A_3 \cdot A_4 \cdot T_{\text{tool}}$$

In [5]:
T = simplify(A1 * A2 * A3 * A4 * Ttool)

print("T_base^tool — Transformación homogénea total:")
display(T)

T_base^tool — Transformación homogénea total:


Matrix([
[cos(theta1 + theta2 + theta3), -sin(theta1 + theta2 + theta3), 0, 0.125*cos(theta1 + theta2) + 0.1*cos(theta1 + theta2 + theta3)],
[sin(theta1 + theta2 + theta3),  cos(theta1 + theta2 + theta3), 0, 0.125*sin(theta1 + theta2) + 0.1*sin(theta1 + theta2 + theta3)],
[                            0,                              0, 1,                                                      0.04 - q4],
[                            0,                              0, 0,                                                              1]])

### Componentes de la transformación total

**Matriz de rotación** (orientación del tool tip respecto a la base):

$$
R_{\text{base}}^{\text{tool}} =
\begin{bmatrix}
\cos(\theta_1+\theta_2+\theta_3) & -\sin(\theta_1+\theta_2+\theta_3) & 0 \\
\sin(\theta_1+\theta_2+\theta_3) &  \cos(\theta_1+\theta_2+\theta_3) & 0 \\
0 & 0 & 1
\end{bmatrix}
$$

**Vector de posición** (posición del tool tip respecto a la base):

$$
p_{\text{base}}^{\text{tool}} =
\begin{bmatrix}
L_1 \cos(\theta_1+\theta_2) + L_2 \cos(\theta_1+\theta_2+\theta_3) \\
L_1 \sin(\theta_1+\theta_2) + L_2 \sin(\theta_1+\theta_2+\theta_3) \\
H_1 + (H_2 - q_4) - \text{Tool}
\end{bmatrix}
$$

---
## 5. Verificación

Evaluamos la cinemática directa para una configuración conocida y verificamos
que el resultado sea físicamente coherente con la geometría del URDF.

**Caso de prueba:** $\theta_1 = \pi$, $\theta_2 = \pi$, $\theta_3 = \pi$, $q_4 = 0.030\,\text{m}$

Con todos los ángulos en $\pi$, el brazo debería estar plegado sobre sí mismo
(configuración "folded"). La coordenada Z debe reflejar que el efector se ha
extendido 0.030 m hacia abajo desde su posición neutra.

In [6]:
T_eval = T.subs({
    theta1: pi,
    theta2: pi,
    theta3: pi,
    q4: 0.030
})

print("T_base^tool evaluada en θ₁=π, θ₂=π, θ₃=π, q₄=0.030:")
display(T_eval)

print(f"\nPosición del tool tip:")
px = float(T_eval[0, 3])
py = float(T_eval[1, 3])
pz = float(T_eval[2, 3])
print(f"  x = {px:.4f} m")
print(f"  y = {py:.4f} m")
print(f"  z = {pz:.4f} m")
print(f"\n  → posición: ({px:.4f}, {py:.4f}, {pz:.4f})")
print(f"\n  El brazo plegado queda en x≈{px:.3f} (cerca del origen) "
      f"y el efector bajó {0.030:.3f} m desde su altura neutra.")

T_base^tool evaluada en θ₁=π, θ₂=π, θ₃=π, q₄=0.030:


Matrix([
[-1,  0, 0, 0.025],
[ 0, -1, 0,     0],
[ 0,  0, 1,  0.01],
[ 0,  0, 0,     1]])


Posición del tool tip:
  x = 0.0250 m
  y = 0.0000 m
  z = 0.0100 m

  → posición: (0.0250, 0.0000, 0.0100)

  El brazo plegado queda en x≈0.025 (cerca del origen) y el efector bajó 0.030 m desde su altura neutra.


---
## Resumen

1. **Sistemas de coordenadas** definidos siguiendo la convención DH estándar para
   un robot SCARA de 4 GDL (3 revolutas + 1 prismática).
2. **Tabla DH** generada con los parámetros extraídos del URDF, considerando
   que el joint prismático axis_3 tiene eje $(0,0,-1)$.
3. **Matrices individuales** $A_1$ a $A_4$ y $T_{\text{tool}}$ construidas.
4. **Matriz de transformación homogénea total** obtenida y simplificada.
5. **Verificación numérica** validada sobre una configuración de prueba.

El modelo cinemático está listo para ser usado en etapas posteriores
(cinemática inversa, control de movimiento, simulación, etc.)